# Phase 3.1 : Chargement MobileNetV2 et freezing

Chats vs chiens (Microsoft Cats vs Dogs Dataset). Suite de `phase2_3_comparaison.ipynb`.

**Ce notebook est autonome** : la section "Reprise" ci-dessous refait le setup des phases 1.1 et 1.2 (téléchargement, organisation), avec un pipeline de données adapté à MobileNetV2 (160x160, `preprocess_input` au lieu de `Rescaling`).

## Reprise (setup phases 1.1 et 1.2 (pipeline adapte a MobileNetV2), condensé)

Nécessite un `kaggle.json` (voir phase 1.1 pour l'obtenir).

In [ ]:
CLASS_A = "cat"
CLASS_B = "dog"
DATA_ROOT = "/content/data"

In [ ]:
!pip install kaggle -q
!mkdir -p ~/.kaggle

In [ ]:
from google.colab import files
files.upload()  # sélectionner kaggle.json

import os
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

In [ ]:
!kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset -p /content/data/
!cd /content/data && unzip -q microsoft-catsvsdogs-dataset.zip

In [ ]:
import os, shutil, random
import tensorflow as tf

RAW_DIR = os.path.join(DATA_ROOT, "PetImages")
raw_dirs = {CLASS_A: os.path.join(RAW_DIR, "Cat"), CLASS_B: os.path.join(RAW_DIR, "Dog")}


@tf.function
def _try_decode(img_bytes):
    # Force l'execution en mode graphe (via tf.function), qui est plus stricte que
    # l'execution eager sur certains formats (ex. BMP mal forme deguise en .jpg) et
    # reproduit exactement le mode d'execution utilise par image_dataset_from_directory
    # pendant l'entrainement.
    return tf.io.decode_image(img_bytes, channels=3, expand_animations=False)


def list_valid_images(folder):
    # Ni PIL.Image.verify() ni la verification d'entete JFIF ni tf.io.decode_image() en
    # mode eager ne suffisent : ce dataset contient des fichiers que seul le decodeur en
    # mode graphe rejette (InvalidArgumentError, "Unknown image file format" ou erreurs
    # de dimension BMP). On teste donc chaque fichier via une tf.function.
    valid = []
    for fname in os.listdir(folder):
        path = os.path.join(folder, fname)
        if os.path.getsize(path) == 0:
            continue
        try:
            img_bytes = tf.io.read_file(path)
            _try_decode(img_bytes)
        except Exception:
            continue
        valid.append(path)
    return valid


files_a = list_valid_images(raw_dirs[CLASS_A])
files_b = list_valid_images(raw_dirs[CLASS_B])

for split in ["train", "val"]:
    for cls in [CLASS_A, CLASS_B]:
        os.makedirs(os.path.join(DATA_ROOT, split, cls), exist_ok=True)

random.seed(42)
random.shuffle(files_a)
random.shuffle(files_b)


def split_and_copy(file_list, cls):
    cut = int(len(file_list) * 0.8)
    train_files, val_files = file_list[:cut], file_list[cut:]
    for f in train_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "train", cls, os.path.basename(f)))
    for f in val_files:
        shutil.copy(f, os.path.join(DATA_ROOT, "val", cls, os.path.basename(f)))


split_and_copy(files_a, CLASS_A)
split_and_copy(files_b, CLASS_B)
print(f"{CLASS_A}: {len(files_a)} images valides")
print(f"{CLASS_B}: {len(files_b)} images valides")

In [ ]:
IMG_SIZE_TL = (160, 160)
BATCH_SIZE = 32

train_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "train"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42,
)

val_ds_tl = tf.keras.utils.image_dataset_from_directory(
    os.path.join(DATA_ROOT, "val"),
    labels="inferred",
    label_mode="binary",
    class_names=[CLASS_A, CLASS_B],
    color_mode="rgb",
    image_size=IMG_SIZE_TL,
    batch_size=BATCH_SIZE,
    shuffle=False,
    seed=42,
)

# preprocess_input de MobileNetV2 normalise dans [-1, 1] (pas [0, 1]) : pas de
# Rescaling(1./255) ici, preprocess_input le remplace.
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_tl = train_ds_tl.map(lambda x, y: (preprocess_input(x), y))
val_ds_tl = val_ds_tl.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds_tl = train_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)
val_ds_tl = val_ds_tl.cache().prefetch(buffer_size=AUTOTUNE)

## Phase 3.1 : Chargement MobileNetV2 et freezing

**Objectif** : charger MobileNetV2 pré-entraîné sur ImageNet, geler la base, construire une tête de classification custom, et vérifier dans `model_tl.summary()` que zéro paramètre de la base n'est entraînable.

MobileNetV2 attend des images de taille minimum 32x32 ; la taille recommandée est 160x160 ou 224x224 — on a reconstruit le pipeline avec `IMG_SIZE_TL=(160,160)` dans la Reprise ci-dessus.

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(160, 160, 3),
    include_top=False,
    weights="imagenet",
)

print(f"Nombre de couches dans la base : {len(base_model.layers)}")
print(f"Nombre de parametres totaux : {base_model.count_params():,}")

base_model.trainable = False

In [ ]:
inputs = tf.keras.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False)  # training=False : BatchNorm reste en mode inference
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model_tl = tf.keras.Model(inputs, outputs)
model_tl.summary()

### Vérifications avant la phase 3.2

- **Happy path** : `model_tl.summary()` montre la base MobileNetV2 gelée (`Non-trainable params` ≈ 2.2M), `Trainable params` autour de **165 000** pour la tête custom (`1280*128+128 + 128*1+1`), output shape finale `(None, 1)`.
- **Edge case** : charger MobileNetV2 sans `weights="imagenet"` donnerait des poids aléatoires — le transfer learning n'apporterait plus rien (à comparer en phase 3.2 si tu es curieux : `val_accuracy` après quelques epochs serait bien plus basse).
- **Adversarial** : `include_top=True` avec une tête custom par-dessus lèverait une erreur de shape (la tête ImageNet sort 1000 classes, pas l'embedding attendu) — `include_top=False` est obligatoire dès qu'on ajoute une tête custom.

**Prochaine étape** : Phase 3.2 — entraîner uniquement la tête de classification (base gelée).